# Housing Price Prediction Notebook
## This notebook cleans, preprocesses, and trains a machine learning model to predict property sale prices using the Housing Dataset.
### Import Required Libraries

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb

# Load Datasets

In [8]:
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Save IDs for submission file generation

In [9]:
test_ids = test_df['Id']

# Explicitly Define Column Mappings & Orderings
# Correct numeric misclassifications: MSSubClass represents types of dwellings, not quantities

In [10]:
train_df['MSSubClass'] = train_df['MSSubClass'].astype(str)
test_df['MSSubClass'] = test_df['MSSubClass'].astype(str)

In [11]:
structural_na_cols = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 
    'BsmtFinType2', 'FireplaceQu', 'GarageType', 'GarageFinish', 
    'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature'
]

# Standard ordinal scales present in the dataset (ordered worst to best)

In [12]:
qual_scale_1 = ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex']
qual_scale_2 = ['NA', 'No', 'Mn', 'Av', 'Gd']
qual_scale_3 = ['NA', 'Unf', 'LwQ', 'Rec', 'BLQ', 'ALQ', 'GLQ']
qual_scale_4 = ['NA', 'Unf', 'RFn', 'Fin']
qual_scale_5 = ['NA', 'MnWw', 'GdWo', 'MnPrv', 'GdPrv']

ordinal_mappings = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual': qual_scale_1,
    'BsmtCond': qual_scale_1,
    'BsmtExposure': qual_scale_2,
    'BsmtFinType1': qual_scale_3,
    'BsmtFinType2': qual_scale_3,
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': qual_scale_1,
    'GarageFinish': qual_scale_4,
    'GarageQual': qual_scale_1,
    'GarageCond': qual_scale_1,
    'PoolQC': qual_scale_1,
    'Fence': qual_scale_5
}

# Separate variables into feature processing arrays

In [13]:
ordinal_cols = list(ordinal_mappings.keys())
nominal_cols = [col for col in train_df.select_dtypes(include=['object']).columns if col not in ordinal_cols]
numeric_cols = [col for col in train_df.select_dtypes(exclude=['object']).columns if col not in ['Id', 'SalePrice']]

# Structural Cleaning (Handling Pseudo-NAs)
# Fill structural string flags globally so encoders don't drop them

In [14]:
for df in [train_df, test_df]:
    df[structural_na_cols] = df[structural_na_cols].fillna('NA')

# Feature Preprocessing Pipelines
# 1. Numeric Feature Imputer & Scaler

In [15]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

# 2. Ordinal Feature Processing

In [16]:
ordinal_categories = [ordinal_mappings[col] for col in ordinal_cols]
ordinal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
    ('ordinal', OrdinalEncoder(categories=ordinal_categories, handle_unknown='use_encoded_value', unknown_value=-1))
])

# 3. Nominal Feature Processing (Target Encoding inside XGBoost native routine via pipeline compatibility)

In [17]:
nominal_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Combine transformers into unique structural ColumnTransformer

In [18]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('ord', ordinal_transformer, ordinal_cols),
        ('nom', nominal_transformer, nominal_cols)
    ])

# Set Up Train/Test Splits and Log-Transform Target

In [19]:
X_train = train_df.drop(columns=['Id', 'SalePrice'])
y_train = np.log1p(train_df['SalePrice'])
X_test = test_df.drop(columns=['Id'])

# Create and Validate the Unified Pipeline
# Using XGBoost Regressor tuned to handle complex tabular dependencies safely

In [20]:
model = xgb.XGBRegressor(
    n_estimators=350,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

clf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', model)
])

# Cross-Validation to gauge root-mean-squared log error performance safely

In [26]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(clf_pipeline, X_train, y_train, scoring='neg_root_mean_squared_error', cv=kf)
rmse_scores = -cv_scores

# Train Final Model & Generate Prediction Submission File
# Train on the entire available feature matrix

In [22]:
clf_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('ord', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


# Predict on test data and invert log-transformations using exponential matching

In [23]:
log_predictions = clf_pipeline.predict(X_test)
final_predictions = np.expm1(log_predictions)

# Construct Output DataFrame matching the sample structure requirements perfectly

In [24]:
submission_df = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': final_predictions
})

# Save predictions output

In [25]:
submission_df.to_csv("house_price_predictions.csv", index=False)
print("Predictions saved to 'house_price_predictions.csv' successfully!")

Predictions saved to 'house_price_predictions.csv' successfully!
